In [12]:
# Cell 1: Imports & Load Data
import json
import re
import random
from pathlib import Path
from collections import Counter
from mlx_lm import load, generate

PROJECT = Path.home() / "llm-mail-trainer"

# Load all parsed emails (40k)
with open(PROJECT / "data/parsed/emails.json", 'r') as f:
    all_emails = json.load(f)

# Load discovered patterns
with open(PROJECT / "data/filtered/finance_patterns.json", 'r') as f:
    patterns = json.load(f)

print(f"✅ Loaded {len(all_emails):,} emails")
print(f"✅ Loaded patterns:")
print(f"   Senders: {patterns['finance_senders']}")
print(f"   Keywords: {patterns['finance_keywords']}")

# Cell 2: Filter full dataset using discovered patterns
def is_finance_email(email):
    """Check if email is finance-related using discovered patterns."""
    
    sender = email['sender'].lower()
    subject = email['subject'].lower()
    body = email['body'].lower()
    combined = f"{sender} {subject} {body}"
    
    # Check sender patterns
    for s in patterns['finance_senders']:
        if s in sender:
            return True
    
    # Check keywords in subject or body
    for kw in patterns['finance_keywords']:
        if kw in combined:
            return True
    
    return False

# Apply filter to all emails
finance_emails = [e for e in all_emails if is_finance_email(e)]

print(f"Total emails: {len(all_emails):,}")
print(f"Finance emails found: {len(finance_emails):,}")
print(f"Percentage: {len(finance_emails)/len(all_emails)*100:.1f}%")

# Show sample
print(f"\n=== SAMPLE FINANCE EMAILS ===")
for i, email in enumerate(finance_emails[:5]):
    print(f"{i+1}. {email['subject'][:60]}")

# Cell 3: Stricter filter - transaction emails only
def is_transaction_email(email):
    """Check if email contains actual transaction data."""
    
    body = email['body'].lower()
    subject = email['subject'].lower()
    combined = f"{subject} {body}"
    
    # Must have transaction indicators
    has_transaction = any(kw in combined for kw in ['debited', 'credited', 'payment of', 'transferred'])
    
    # Must have amount pattern (Rs. or ₹)
    has_amount = bool(re.search(r'(?:rs\.?|₹)\s*[\d,]+', combined))
    
    # Must have account reference
    has_account = bool(re.search(r'(?:account|a/c|ac no)', combined))
    
    return has_transaction and has_amount

# Apply stricter filter
transaction_emails = [e for e in all_emails if is_transaction_email(e)]

print(f"Transaction emails found: {len(transaction_emails):,}")

# Show samples
print(f"\n=== SAMPLE TRANSACTION EMAILS ===")
for i, email in enumerate(transaction_emails[:5]):
    print(f"\n{i+1}. Subject: {email['subject'][:70]}")
    print(f"   Body: {email['body'][:150]}...")

# Cell 4: Entity extraction function
def extract_entities(text):
    """Extract financial entities from email text."""
    
    entities = {}
    
    # Amount
    amount_match = re.search(r'(?:Rs\.?|₹)\s*([\d,]+(?:\.\d{2})?)', text)
    if amount_match:
        entities['amount'] = amount_match.group(1).replace(',', '')
    
    # Type
    if 'debited' in text.lower():
        entities['type'] = 'debit'
    elif 'credited' in text.lower():
        entities['type'] = 'credit'
    
    # Account
    account_match = re.search(r'(?:account|A/C|a/c)\s*[:\s]?\s*(\w+)', text, re.IGNORECASE)
    if account_match:
        entities['account'] = account_match.group(1)
    
    # Date
    date_match = re.search(r'(\d{2}-\d{2}-\d{2,4})', text)
    if date_match:
        entities['date'] = date_match.group(1)
    
    # Reference
    ref_match = re.search(r'reference\s*(?:number|no\.?)?\s*(?:is)?\s*(\d+)', text, re.IGNORECASE)
    if ref_match:
        entities['reference'] = ref_match.group(1)
    
    return entities

# Test
sample = transaction_emails[0]
print(f"Subject: {sample['subject'][:60]}")
print(f"Extracted: {extract_entities(sample['body'])}")

# Cell 5: Create training data for fine-tuning
training_data = []

for email in transaction_emails:
    entities = extract_entities(email['body'])
    
    # Skip if no entities extracted
    if len(entities) < 2:
        continue
    
    # Create training example
    example = {
        "prompt": f"Extract financial entities from this email:\n\nSubject: {email['subject']}\n\nBody: {email['body'][:1500]}",
        "completion": json.dumps(entities, indent=2)
    }
    
    training_data.append(example)

print(f"Training examples created: {len(training_data)}")

# Show sample
print(f"\n=== SAMPLE TRAINING EXAMPLE ===")
print(f"PROMPT:\n{training_data[0]['prompt'][:300]}...")
print(f"\nCOMPLETION:\n{training_data[0]['completion']}")

# Cell 6: Split and save training data
random.seed(42)
random.shuffle(training_data)

# Split: 90% train, 10% validation
split_idx = int(len(training_data) * 0.9)
train_data = training_data[:split_idx]
valid_data = training_data[split_idx:]

print(f"Train set: {len(train_data)}")
print(f"Validation set: {len(valid_data)}")

# Save as JSONL (required format for MLX)
training_dir = PROJECT / "data/training"
training_dir.mkdir(exist_ok=True)

# Save train.jsonl
with open(training_dir / "train.jsonl", 'w') as f:
    for example in train_data:
        f.write(json.dumps(example) + '\n')

# Save valid.jsonl
with open(training_dir / "valid.jsonl", 'w') as f:
    for example in valid_data:
        f.write(json.dumps(example) + '\n')

print(f"\n✅ Saved to {training_dir}/")
print(f"   train.jsonl ({len(train_data)} examples)")
print(f"   valid.jsonl ({len(valid_data)} examples)")

# Cell 7: Fine-tune configuration
print("=== FINE-TUNING SETUP ===")
print(f"Base model: {PROJECT / 'models/base/phi3-mini'}")
print(f"Training data: {PROJECT / 'data/training/train.jsonl'}")
print(f"Validation data: {PROJECT / 'data/training/valid.jsonl'}")
print(f"Output: {PROJECT / 'models/adapters/finance-lora'}")

print("\n⚠️ Fine-tuning will take 1-2 hours")
print("Run the following command in Terminal (not in notebook):")

command = f"""
cd {PROJECT}
source venv/bin/activate

mlx_lm.lora \\
    --model models/base/phi3-mini \\
    --data data/training \\
    --train \\
    --batch-size 1 \\
    --lora-layers 8 \\
    --iters 500 \\
    --adapter-path models/adapters/finance-lora
"""

print(command)

# Cell 8: Test fine-tuned model


# Load base model with fine-tuned adapter
model_path = str(PROJECT / "models/base/phi3-mini")
adapter_path = str(PROJECT / "models/adapters/finance-lora")

print("Loading fine-tuned model...")
model, tokenizer = load(model_path, adapter_path=adapter_path)
print("✅ Model loaded with LoRA adapter")

# Cell 9: Test entity extraction on sample email
test_email = """
Dear Customer, Rs.2500.00 has been debited from account 3545 to VPA swiggy@ybl 
for Swiggy order on 28-12-25. Your UPI transaction reference number is 534567891234. 
If you did not authorize this transaction, please call 1800-XXX-XXXX.
"""

prompt = f"Extract financial entities from this email:\n\n{test_email}"

print("=== INPUT EMAIL ===")
print(test_email)
print("\n=== MODEL OUTPUT ===")

response = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=200,
    verbose=False
)

print(response)

# Cell 10: Test on credit transaction
test_email_2 = """
Dear Customer, Rs.45,000.00 has been credited to your account 7890 
on 27-12-25. Salary from ACME CORP. Reference: NEFT123456789.
Available balance: Rs.52,340.00
"""

prompt = f"Extract financial entities from this email:\n\n{test_email_2}"

print("=== INPUT EMAIL ===")
print(test_email_2)
print("\n=== MODEL OUTPUT ===")

response = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=200,
    verbose=False
)

print(response)

# Cell 11: Test credit transaction with better prompt
test_email_credit = """Subject: Amount Credited to Your Account

Dear Customer, Rs.45,000.00 has been credited to your account 7890 
on 27-12-25. Salary from ACME CORP. Reference: NEFT123456789.
Available balance: Rs.52,340.00"""

prompt = f"""Extract financial entities from this email:

Subject: Amount Credited to Your Account

Body: {test_email_credit}"""

print("=== INPUT ===")
print(test_email_credit)
print("\n=== MODEL OUTPUT ===")

response = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=200,
    verbose=False
)

print(response)

# Cell 12: Merge LoRA adapter with base model
import shutil

# Create merged model directory
merged_path = PROJECT / "models/merged/finance-llm"
merged_path.mkdir(parents=True, exist_ok=True)

print("To merge the model, run this in Terminal:")
print()
merge_command = f"""cd {PROJECT}
source venv/bin/activate

mlx_lm.fuse \\
    --model models/base/phi3-mini \\
    --adapter-path models/adapters/finance-lora \\
    --save-path models/merged/finance-llm
"""
print(merge_command)

✅ Loaded 40,820 emails
✅ Loaded patterns:
   Senders: ['icici', 'hdfc', 'groww', 'zerodha', 'paisabazaar', 'sbi', 'axis', 'kotak']
   Keywords: ['debited', 'credited', 'transaction', 'upi', 'a/c', 'balance', 'payment']
Total emails: 40,820
Finance emails found: 8,116
Percentage: 19.9%

=== SAMPLE FINANCE EMAILS ===
1. Inquiry from Robokits India
2. 🍩 Doodles, donuts & data to close out 2025
3. Daily Equity Margin Statement for HTV475 - December 23, 2025
4. How Financially Ready Are You? Take the Quiz Now!
5. 📈 Dear Ranjit Behera, diversify beyond just savings
Transaction emails found: 598

=== SAMPLE TRANSACTION EMAILS ===

1. Subject: ❗ You have done a UPI txn. Check details!
   Body: HDFC BANK Dear Customer, Rs.50000.00 has been debited from account 3545 to VPA subhashreebadatya250@okicici SUBHASHREE BADATYA on 22-12-25. Your UPI t...

2. Subject: ❗ You have done a UPI txn. Check details!
   Body: HDFC BANK Dear Customer, Rs.50000.00 has been debited from account 3545 to VPA subhashr